# Recomendador de películas con grafos · Grupo 4

**Curso:** 1ACC0184 Complejidad Algorítmica · UPC · ciclo 2026-20 · sección 4755
**Profesor:** Abraham Sopla Maluscán
**Caso de estudio 4:** Recomendador de contenido (streaming) sobre **MovieLens 1M**

| Integrante | Código | Rango de usuarios |
|---|---|---|
| Rojas Gómez, Valeria | U202411373 | 1 a 2013 |
| Rivas Castillo, Christoper | U202323551 | 2014 a 4026 |
| Carhuaz Centeno, Briguite | U20241D932 | 4027 a 6040 |

Este cuaderno reúne **todo el código del trabajo**, en el mismo orden que el informe:

| Parte | Qué hace | Sección del informe | Hito |
|---|---|---|---|
| 1 | Descarga y carga de MovieLens 1M | Descripción del conjunto de datos | TB1 |
| 2 | Exploración del dataset (cifras y gráficos) | Descripción del conjunto de datos | TB1 |
| 3 | Grafo bipartito usuario–película (lista de adyacencia propia) | Relación con grafos | TB1 |
| 4 | Aristas usuario–usuario (proyección del grafo) | Relación con grafos | TB1 |
| 5 | Figuras del grafo y subgrafos por integrante | Visualización | TB1 |
| 6 | BFS de 3 niveles + selección voraz Top-K con heap | Propuesta / Diseño del aplicativo | TB2 |
| 7 | Fuerza bruta como línea base | Diseño del aplicativo | TB2 |
| 8 | UFDS + Kruskal: comunidades de gustos | Propuesta / Diseño del aplicativo | TB2 |
| 9 | Validación: pruebas, precisión y tiempos | Validación de resultados y pruebas | Final |
| 10 | Interfaz gráfica (GUI) dentro de Colab | Diseño del aplicativo / exposición | Final |
| 11 | Cifras para el informe y descarga de resultados | Todas | Todos |

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todas*. Tarda unos minutos con el dataset completo. Al final se descarga `resultados_grupo4.zip` con todas las figuras y tablas.

Los algoritmos (grafo, BFS, voraz, fuerza bruta, UFDS, Kruskal) están **escritos a mano** con la biblioteca estándar de Python. `networkx` solo se usa para **dibujar** los subgrafos, y `numpy` para contar películas en común entre usuarios.

## Parte 0. Librerías y parámetros

In [ ]:
# Celda 0.1: librerías
# Biblioteca estándar (no hay que instalar nada)
import os, time, random, zipfile, urllib.request, heapq
from collections import defaultdict, deque, Counter

# Librerías externas (Google Colab ya las trae instaladas)
import numpy as np                  # matrices: cuenta de películas en común entre usuarios
import pandas as pd                 # leer los .dat y resumir tablas
import matplotlib.pyplot as plt     # gráficos
from matplotlib.lines import Line2D # leyendas personalizadas
import networkx as nx               # SOLO para dibujar subgrafos pequeños

plt.rcParams["figure.dpi"] = 110
print("numpy", np.__version__, "| pandas", pd.__version__, "| networkx", nx.__version__)

In [ ]:
# Celda 0.2: parámetros del trabajo (cámbialos aquí y vuelve a ejecutar todo)
USAR_DATOS_SINTETICOS = False  # True SOLO para probar el cuaderno sin internet; para el informe, False

GUSTO = 4            # una calificación >= GUSTO cuenta como "le gustó"
S = 20               # arista usuario–usuario si ambos dieron >= GUSTO a por lo menos S películas en común
K = 10               # cuántas películas recomienda el sistema (Top-K)
MAX_VECINOS = 50     # BFS: cuántos usuarios parecidos se conservan en el nivel 2
USUARIO_DEMO = 1     # usuario que se usa en los ejemplos
TAMANO_MAX_COMUNIDAD = 300  # Kruskal no une dos grupos si juntos pasarían de este tamaño
MIN_TAMANO_COMUNIDAD = 10   # grupos más pequeños no se reportan como comunidad
SEMILLA = 42         # semilla fija para que todo sea reproducible

# Rango de usuarios asignado a cada integrante (para el mínimo de 500 nodos por integrante)
RANGOS = {"Rojas Gómez": (1, 2013), "Rivas Castillo": (2014, 4026), "Carhuaz Centeno": (4027, 6040)}

# Parámetros de las figuras 1 y 2 (los mismos del Colab de figuras)
N_PELICULAS = 12             # películas más calificadas que aparecen en la Figura 1
USUARIOS_POR_INTEGRANTE = 4  # usuarios de cada rango en la Figura 1
MIN_PELICULAS_VISTAS = 6     # cada usuario de la figura calificó al menos 6 de esas 12 películas

CARPETA = "ml-1m"
SALIDA = "resultados"
os.makedirs(SALIDA, exist_ok=True)
random.seed(SEMILLA)
np.random.seed(SEMILLA)

## Parte 1. Conjunto de datos: descarga y carga

MovieLens 1M (GroupLens Research, Universidad de Minnesota) tiene tres archivos de texto separados por `::`:

| Archivo | Columnas | Filas |
|---|---|---|
| `ratings.dat` | UserID, MovieID, Rating (1–5), Timestamp | 1,000,209 |
| `users.dat` | UserID, Gender, Age, Occupation, Zip-code | 6,040 |
| `movies.dat` | MovieID, Title, Genres (separados por `\|`) | 3,883 |

El dataset **no se sube al repositorio** porque su licencia no permite redistribuirlo; esta celda lo descarga de la página oficial.

In [ ]:
# Celda 1.1: descargar MovieLens 1M (o generar datos de prueba con el mismo formato)
URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"


def generar_datos_sinteticos(carpeta, n_usuarios=6040, n_peliculas=3706, semilla=SEMILLA):
    # Datos INVENTADOS con el mismo formato que MovieLens 1M. Solo sirven para comprobar
    # que el cuaderno corre sin internet; NUNCA se usan para el informe.
    rng = np.random.default_rng(semilla)
    generos = ["Action", "Adventure", "Animation", "Children's", "Comedy", "Crime", "Documentary", "Drama",
               "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"]
    os.makedirs(carpeta, exist_ok=True)
    ids_m = np.sort(rng.choice(np.arange(1, 3953), n_peliculas, replace=False))
    gen_m = [list(rng.choice(len(generos), rng.integers(1, 4), replace=False)) for _ in ids_m]
    with open(f"{carpeta}/movies.dat", "w", encoding="latin-1") as f:
        for m, gs in zip(ids_m, gen_m):
            f.write(f"{m}::Pelicula {m} ({rng.integers(1930, 2001)})::{'|'.join(generos[g] for g in gs)}\n")
    with open(f"{carpeta}/users.dat", "w", encoding="latin-1") as f:
        for u in range(1, n_usuarios + 1):
            f.write(f"{u}::{rng.choice(['M', 'F'])}::{rng.choice([1, 18, 25, 35, 45, 50, 56])}::{rng.integers(0, 21)}::{rng.integers(10000, 99999)}\n")
    popularidad = 1 / np.arange(1, n_peliculas + 1) ** 0.8          # pocas películas muy vistas (cola larga)
    rng.shuffle(popularidad)
    grupo_gustos = [set(rng.choice(len(generos), 3, replace=False)) for _ in range(8)]  # 8 "gustos" ocultos
    with open(f"{carpeta}/ratings.dat", "w", encoding="latin-1") as f:
        for u in range(1, n_usuarios + 1):
            prefer = grupo_gustos[rng.integers(0, 8)]
            afin = np.array([len(prefer & set(g)) for g in gen_m])
            p = popularidad * (1 + 3 * afin)
            n = int(min(n_peliculas, rng.integers(20, 200)))
            for j in rng.choice(n_peliculas, n, replace=False, p=p / p.sum()):
                r = int(np.clip(round(rng.normal(3 + 0.8 * afin[j], 0.9)), 1, 5))
                f.write(f"{u}::{ids_m[j]}::{r}::{978300000 + int(rng.integers(0, 3 * 10**7))}\n")


if USAR_DATOS_SINTETICOS:
    CARPETA = "ml-1m-sintetico"
    if not os.path.exists(f"{CARPETA}/ratings.dat"):
        generar_datos_sinteticos(CARPETA)
    print("ATENCIÓN: datos sintéticos de prueba. Para el informe pon USAR_DATOS_SINTETICOS = False.")
elif not os.path.exists(f"{CARPETA}/ratings.dat"):
    try:
        urllib.request.urlretrieve(URL, "ml-1m.zip")
    except Exception as e:  # si la descarga falla, se puede subir el zip a mano
        print("No se pudo descargar:", e, "\nSube ml-1m.zip (descargado de grouplens.org) en la ventana que aparece.")
        from google.colab import files
        files.upload()
    zipfile.ZipFile("ml-1m.zip").extractall(".")
print("Archivos:", sorted(os.listdir(CARPETA)))

In [ ]:
# Celda 1.2: leer los tres archivos como tablas (DataFrames)
# sep="::" necesita engine="python"; los archivos vienen en latin-1 (tildes de los títulos)
leer = dict(sep="::", engine="python", encoding="latin-1")
ratings = pd.read_csv(f"{CARPETA}/ratings.dat", names=["UserID", "MovieID", "Rating", "Timestamp"], **leer)
users = pd.read_csv(f"{CARPETA}/users.dat", names=["UserID", "Gender", "Age", "Occupation", "Zip"], **leer)
movies = pd.read_csv(f"{CARPETA}/movies.dat", names=["MovieID", "Title", "Genres"], **leer)

# Diccionarios para traducir id -> título y id -> lista de géneros en O(1)
titulo = dict(zip(movies.MovieID, movies.Title))
generos_de = {m: g.split("|") for m, g in zip(movies.MovieID, movies.Genres)}

print(f"Calificaciones: {len(ratings):,} | Usuarios: {len(users):,} | Películas en el catálogo: {len(movies):,}")
ratings.head()

## Parte 2. Exploración del conjunto de datos

Cifras y gráficos para la sección *Descripción del conjunto de datos*: cuántas calificaciones hay de cada valor, cuántas hace cada usuario y cuántas recibe cada película.

In [ ]:
# Celda 2.1: estadísticas generales
por_usuario = ratings.groupby("UserID").size()      # grado de cada usuario en el grafo
por_pelicula = ratings.groupby("MovieID").size()    # grado de cada película en el grafo
n_u, n_m = ratings.UserID.nunique(), ratings.MovieID.nunique()

resumen = pd.Series({
    "Usuarios": n_u,
    "Películas en movies.dat": len(movies),
    "Películas con al menos una calificación": n_m,
    "Películas sin calificaciones (nodos aislados)": len(movies) - n_m,
    "Calificaciones": len(ratings),
    "Calificación promedio": round(ratings.Rating.mean(), 3),
    "Calificaciones >= GUSTO (me gusta)": int((ratings.Rating >= GUSTO).sum()),
    "Mínimo de calificaciones por usuario": por_usuario.min(),
    "Mediana de calificaciones por usuario": por_usuario.median(),
    "Máximo de calificaciones por usuario": por_usuario.max(),
    "Mediana de calificaciones por película": por_pelicula.median(),
    "Película más calificada": f"{titulo[por_pelicula.idxmax()]} ({por_pelicula.max():,})",
    "Matriz usuario × película llena (%)": round(100 * len(ratings) / (n_u * n_m), 2),
})
resumen.to_frame("Valor")

In [ ]:
# Celda 2.2: Figura 3, distribución de calificaciones y de grados
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

conteo = ratings.Rating.value_counts().sort_index()
ax[0].bar(conteo.index, conteo.values, color=["#d73027", "#fc8d59", "#bdbdbd", "#91bfdb", "#4575b4"])
ax[0].set(title="Calificaciones por número de estrellas", xlabel="Estrellas", ylabel="Calificaciones")
for x, y in zip(conteo.index, conteo.values):
    ax[0].text(x, y, f"{y:,}", ha="center", va="bottom", fontsize=8)

ax[1].hist(por_usuario, bins=60, color="#2f6fb0")
ax[1].set(title="Calificaciones por usuario (grado del usuario)", xlabel="Calificaciones", ylabel="Usuarios", yscale="log")

ax[2].hist(por_pelicula, bins=60, color="#e08a2c")
ax[2].set(title="Calificaciones por película (grado de la película)", xlabel="Calificaciones", ylabel="Películas", yscale="log")

fig.suptitle("Figura 3. Distribución de las calificaciones en MovieLens 1M", fontsize=12)
fig.tight_layout()
fig.savefig(f"{SALIDA}/figura3_distribucion.png", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# Celda 2.3: géneros más calificados (una película puede tener varios géneros)
generos_calif = Counter()
for m, n in por_pelicula.items():
    for g in generos_de.get(m, []):
        generos_calif[g] += n
pd.Series(generos_calif).sort_values(ascending=False).to_frame("Calificaciones").head(18)

## Parte 3. Grafo bipartito usuario–película

**Modelo.** G = (V, E) no dirigido y ponderado, con V = U ∪ M (usuarios y películas).
Hay una arista (u, m) cuando el usuario u calificó la película m; su peso es la calificación (1 a 5).
Como ninguna arista une dos usuarios ni dos películas, el grafo es **bipartito**.

**Estructura de datos.** Lista de adyacencia con diccionarios, guardada desde los dos lados:

- `adj_usuario[u] = {m: calificación, ...}` (películas que calificó u)
- `adj_pelicula[m] = {u: calificación, ...}` (usuarios que calificaron m)

Así, consultar los vecinos de un nodo cuesta O(grado) y saber si existe una arista cuesta O(1) en promedio.
Construir el grafo recorre cada calificación una vez: **O(|V| + |E|)**.

In [ ]:
# Celda 3.1: clase GrafoBipartito (lista de adyacencia propia, sin librerías)
class GrafoBipartito:
    def __init__(self):
        self.adj_usuario = defaultdict(dict)   # u -> {m: peso}
        self.adj_pelicula = defaultdict(dict)  # m -> {u: peso}

    def agregar_arista(self, u, m, peso):      # O(1)
        self.adj_usuario[u][m] = peso
        self.adj_pelicula[m][u] = peso

    def quitar_arista(self, u, m):             # O(1); se usa en la validación (Parte 9)
        del self.adj_usuario[u][m]
        del self.adj_pelicula[m][u]

    def num_usuarios(self):
        return len(self.adj_usuario)

    def num_peliculas(self):
        return len(self.adj_pelicula)

    def num_nodos(self):
        return self.num_usuarios() + self.num_peliculas()

    def num_aristas(self):                     # cada arista se cuenta una vez (desde el usuario)
        return sum(len(v) for v in self.adj_usuario.values())

    def densidad(self):
        # Densidad de un grafo bipartito: |E| / (|U|·|M|), lo máximo posible es 1
        return self.num_aristas() / (self.num_usuarios() * self.num_peliculas())


# Construcción: una pasada por las 1,000,209 calificaciones -> O(|V| + |E|)
t0 = time.perf_counter()
G = GrafoBipartito()
for u, m, r in zip(ratings.UserID.values, ratings.MovieID.values, ratings.Rating.values):
    G.agregar_arista(int(u), int(m), int(r))
t_grafo = time.perf_counter() - t0

print(f"Nodos activos |V| = {G.num_nodos():,} ({G.num_usuarios():,} usuarios + {G.num_peliculas():,} películas)")
print(f"Nodos si se cuentan las {len(movies) - G.num_peliculas()} películas sin calificaciones: {len(users) + len(movies):,}")
print(f"Aristas |E| = {G.num_aristas():,}")
print(f"Densidad bipartita = {G.densidad():.4f}")
print(f"Tiempo de construcción: {t_grafo:.2f} s")

In [ ]:
# Celda 3.2: nodos por integrante (el enunciado pide >= 500 por integrante y >= 1500 en total)
filas = []
for nombre, (a, b) in RANGOS.items():
    us = [u for u in G.adj_usuario if a <= u <= b]
    ms = {m for u in us for m in G.adj_usuario[u]}          # películas que calificaron esos usuarios
    aristas = sum(len(G.adj_usuario[u]) for u in us)
    filas.append({"Integrante": nombre, "Rango de usuarios": f"{a}–{b}", "Usuarios": len(us),
                  "Películas alcanzadas": len(ms), "Nodos del subgrafo": len(us) + len(ms),
                  "Aristas (calificaciones)": aristas, "Cumple >= 500": len(us) + len(ms) >= 500})
tabla_integrantes = pd.DataFrame(filas)
tabla_integrantes.to_csv(f"{SALIDA}/nodos_por_integrante.csv", index=False)
tabla_integrantes

## Parte 4. Aristas usuario–usuario (proyección del grafo bipartito)

Para detectar **comunidades de gustos similares** (objetivo del caso 4) se derivan aristas entre usuarios a partir de las mismas calificaciones (proyección de un grafo bipartito, Zhou et al., 2007):

> u y v quedan conectados si ambos dieron 4 o 5 estrellas a **por lo menos S = 20 películas en común**; el peso es esa cantidad.

**Cómo se calcula.** A es la matriz usuario × película con A[u, m] = 1 si a u le gustó m. El producto C = A·Aᵀ da, en C[u, v], cuántas películas les gustaron a ambos. Cuesta O(|U|²·|M|) operaciones, pero `numpy` lo hace en segundos porque usa rutinas de álgebra lineal optimizadas.

In [ ]:
# Celda 4.1: matriz de películas en común C = A · Aᵀ
gustos = ratings[ratings.Rating >= GUSTO]
ids_u = np.sort(ratings.UserID.unique())                 # fila i de A  <-> usuario ids_u[i]
ids_m = np.sort(ratings.MovieID.unique())                # columna j de A <-> película ids_m[j]
fila = {u: i for i, u in enumerate(ids_u)}
col = {m: j for j, m in enumerate(ids_m)}

A = np.zeros((len(ids_u), len(ids_m)), dtype=np.float32)
A[gustos.UserID.map(fila).values, gustos.MovieID.map(col).values] = 1
t0 = time.perf_counter()
C = A @ A.T                                              # C[i, j] = películas que les gustaron a ambos
np.fill_diagonal(C, 0)                                   # un usuario no se conecta consigo mismo
t_proy = time.perf_counter() - t0
me_gusta = A.sum(axis=1)                                 # |gustos de u|, para Jaccard en la Parte 8


def aristas_usuario_usuario(s):
    # Devuelve tres arreglos (u, v, peso) con las aristas u–v (u < v) de peso >= s
    iu, iv = np.nonzero(np.triu(C >= s, k=1))
    return ids_u[iu], ids_u[iv], C[iu, iv].astype(int)


uu_a, uu_b, uu_w = aristas_usuario_usuario(S)
print(f"Tiempo de A·Aᵀ: {t_proy:.1f} s")
print(f"Aristas usuario–usuario con S = {S}: {len(uu_w):,}")
print(f"Usuarios con al menos una arista usuario–usuario: {len(set(uu_a) | set(uu_b)):,} de {len(ids_u):,}")
print(f"Peso promedio: {uu_w.mean():.1f} películas en común; máximo: {uu_w.max()}")

## Parte 5. Visualización del grafo

El grafo completo tiene casi 10 mil nodos y un millón de aristas: en una sola imagen sería una mancha. Por eso se dibujan **subgrafos con datos reales** (el enunciado permite "grafo completo y/o subgrafos").

**Criterio de selección.** Las 12 películas más calificadas y, de cada rango de integrante, 4 usuarios al azar (semilla 42) entre los que calificaron al menos 6 de esas películas, para que ningún usuario quede casi sin aristas.

In [ ]:
# Celda 5.1: elegir los nodos de las figuras
top = ratings.MovieID.value_counts().head(N_PELICULAS).index.tolist()
vistas_top = ratings[ratings.MovieID.isin(top)].groupby("UserID").size()
candidatos_fig = vistas_top[vistas_top >= MIN_PELICULAS_VISTAS].index


def elegir(desde, hasta, n, semilla=SEMILLA):
    pool = sorted(u for u in candidatos_fig if desde <= u <= hasta)
    return sorted(random.Random(semilla).sample(pool, min(n, len(pool))))


elegidos = [u for (a, b) in RANGOS.values() for u in elegir(a, b, USUARIOS_POR_INTEGRANTE)]
integrante_de = {u: nombre for nombre, (a, b) in RANGOS.items() for u in elegidos if a <= u <= b}
print("Películas:", [titulo[m] for m in top])
print("Usuarios:", elegidos)

In [ ]:
# Celda 5.2: Figura 1, subgrafo usuario–película (color de la arista = calificación)
COLORES = {1: "#d73027", 2: "#fc8d59", 3: "#bdbdbd", 4: "#91bfdb", 5: "#4575b4"}


def subgrafo(ids_usuarios, peliculas):
    # Copia en networkx solo las aristas del subgrafo (networkx se usa únicamente para dibujar)
    Sg = nx.Graph()
    Sg.add_nodes_from((f"U{u}" for u in ids_usuarios), tipo="usuario")
    Sg.add_nodes_from((f"M{m}" for m in peliculas), tipo="pelicula")
    for u in ids_usuarios:
        for m in peliculas:
            if m in G.adj_usuario[u]:
                Sg.add_edge(f"U{u}", f"M{m}", rating=G.adj_usuario[u][m])
    return Sg


def dibujar(Sg, titulo_figura, archivo):
    us = [n for n, d in Sg.nodes(data=True) if d["tipo"] == "usuario"]
    ms = [n for n, d in Sg.nodes(data=True) if d["tipo"] == "pelicula"]
    pos = nx.bipartite_layout(Sg, us, align="vertical", scale=2)   # usuarios a la izquierda, películas a la derecha
    fig, ax = plt.subplots(figsize=(12, 8), dpi=200)
    for r, c in COLORES.items():
        aristas = [(a, b) for a, b, d in Sg.edges(data=True) if d["rating"] == r]
        nx.draw_networkx_edges(Sg, pos, edgelist=aristas, edge_color=c, width=1.3, alpha=0.85, ax=ax)
    nx.draw_networkx_nodes(Sg, pos, nodelist=us, node_color="#2f6fb0", node_size=450, ax=ax)
    nx.draw_networkx_nodes(Sg, pos, nodelist=ms, node_color="#e08a2c", node_shape="s", node_size=450, ax=ax)
    nx.draw_networkx_labels(Sg, pos, {u: u for u in us}, font_size=6, font_color="white", ax=ax)
    for m in ms:
        x, y = pos[m]
        ax.text(x + 0.08, y, titulo[int(m[1:])][:38], fontsize=7, va="center")
    leyenda = [Line2D([], [], marker="o", ls="", color="#2f6fb0", label="Usuario"),
               Line2D([], [], marker="s", ls="", color="#e08a2c", label="Película")]
    leyenda += [Line2D([], [], color=c, lw=2, label=f"Calificación {r}") for r, c in COLORES.items()]
    ax.legend(handles=leyenda, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=7, fontsize=8, frameon=False)
    ax.set_title(f"{titulo_figura}\n{len(us)} usuarios, {len(ms)} películas, {Sg.number_of_edges()} calificaciones",
                 fontsize=11)
    ax.axis("off")
    ax.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1] + 1.2)
    fig.tight_layout()
    fig.savefig(archivo, bbox_inches="tight")
    plt.show()


S1 = subgrafo(elegidos, top)
dibujar(S1, "Figura 1. Subgrafo usuario–película (MovieLens 1M)", f"{SALIDA}/figura1_usuario_pelicula.png")

In [ ]:
# Celda 5.3: Figura 2, aristas usuario–usuario entre los mismos 12 usuarios
# El peso sale de TODO el dataset (no solo de las 12 películas de la Figura 1)
COLOR_INTEGRANTE = dict(zip(RANGOS, ["#2f6fb0", "#3a9e5c", "#8e5bb5"]))

P = nx.Graph()
P.add_nodes_from(elegidos)
for i, u in enumerate(elegidos):
    for v in elegidos[i + 1:]:
        w = int(C[fila[u], fila[v]])
        if w >= S:
            P.add_edge(u, v, weight=w)

pos = nx.circular_layout(P)
fig, ax = plt.subplots(figsize=(9, 9), dpi=200)
pesos = [d["weight"] for _, _, d in P.edges(data=True)]
maximo = max(pesos, default=1)
nx.draw_networkx_edges(P, pos, width=[0.6 + 4.4 * w / maximo for w in pesos], edge_color="#7a7a7a", alpha=0.7, ax=ax)
nx.draw_networkx_edge_labels(P, pos, {(a, b): d["weight"] for a, b, d in P.edges(data=True)},
                             font_size=6, label_pos=0.5, ax=ax)
nx.draw_networkx_nodes(P, pos, node_color=[COLOR_INTEGRANTE[integrante_de[u]] for u in P.nodes], node_size=900, ax=ax)
nx.draw_networkx_labels(P, pos, {u: f"U{u}" for u in P.nodes}, font_size=7, font_color="white", ax=ax)
leyenda = [Line2D([], [], marker="o", ls="", color=c, markersize=10, label=f"Usuarios {a}–{b}")
           for (nombre, c), (a, b) in zip(COLOR_INTEGRANTE.items(), RANGOS.values())]
leyenda.append(Line2D([], [], color="#7a7a7a", lw=2, label=f"Arista: ≥ {S} películas con {GUSTO}–5 estrellas en común"))
ax.legend(handles=leyenda, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=8, frameon=False)
ax.set_title(f"Figura 2. Aristas usuario–usuario (S = {S})\n"
             f"{P.number_of_nodes()} usuarios, {P.number_of_edges()} aristas; el número es el peso", fontsize=11)
ax.axis("off")
fig.tight_layout()
fig.savefig(f"{SALIDA}/figura2_usuario_usuario.png", bbox_inches="tight")
plt.show()
print(f"Densidad de la Figura 2: {nx.density(P):.2f} (1.0 = todos conectados con todos)")

In [ ]:
# Celda 5.4: un subgrafo usuario–película por integrante, con 8 usuarios de su propio rango
for i, (nombre, (a, b)) in enumerate(RANGOS.items(), start=1):
    Sg = subgrafo(elegir(a, b, 8), top[:10])
    dibujar(Sg, f"Subgrafo de {nombre} (usuarios {a} a {b})", f"{SALIDA}/subgrafo_integrante{i}.png")

## Parte 6. Recomendación: BFS de 3 niveles + selección voraz Top-K

**Idea.** "A usuarios con gustos parecidos a los tuyos les gustaron estas películas".

**BFS acotado a 3 niveles** desde el usuario objetivo u (cola FIFO, `deque`):

| Nivel | Nodos | Qué se hace |
|---|---|---|
| 0 | u | Punto de partida |
| 1 | Películas que a u le gustaron (≥ 4) | Se encolan |
| 2 | Otros usuarios que también les dieron ≥ 4 | similitud(v) = películas que les gustaron a ambos; se conservan los `MAX_VECINOS` más similares |
| 3 | Películas que a esos vecinos les gustaron y u **no** ha visto | puntaje(m) = Σ similitud(v) · calificación(v, m) |

**Complejidad.** Cada nodo y arista alcanzados en 3 saltos se procesa una vez: O(|V'| + |E'|) sobre ese subgrafo; en el peor caso O(|V| + |E|), igual que un BFS completo. Quedarse con los N vecinos cuesta O(|V₂| log N) con un heap.

**Selección voraz Top-K.** De los C candidatos del nivel 3 se toman los K de mayor puntaje con un **min-heap de tamaño K**: cada candidato entra y, si hay más de K, sale el peor. Cuesta **O(C log K)**, menos que ordenar todo, O(C log C). Es voraz porque en cada paso se queda con la mejor opción local (el mayor puntaje) sin reconsiderarla.

In [ ]:
# Celda 6.1: BFS acotado a 3 niveles
def bfs_candidatos(grafo, usuario, umbral=GUSTO, max_vecinos=MAX_VECINOS):
    # Devuelve (puntaje de cada película candidata, vecinos elegidos con su similitud, estadísticas)
    vistas = grafo.adj_usuario.get(usuario, {})
    cola = deque([("U", usuario, 0)])        # (tipo de nodo, id, nivel)
    visitado = {("U", usuario)}
    similitud = defaultdict(int)             # vecino v -> películas que les gustaron a ambos
    puntaje = defaultdict(float)             # película candidata -> puntaje
    vecinos = None                           # se fija al terminar el nivel 1
    stats = Counter()

    while cola:
        tipo, x, nivel = cola.popleft()
        stats["nodos"] += 1

        if nivel == 0:                       # usuario objetivo -> películas que le gustaron
            for m, w in grafo.adj_usuario[x].items():
                stats["aristas"] += 1
                if w >= umbral and ("M", m) not in visitado:
                    visitado.add(("M", m))
                    cola.append(("M", m, 1))

        elif nivel == 1:                     # película -> otros usuarios a los que también les gustó
            for v, w in grafo.adj_pelicula[x].items():
                stats["aristas"] += 1
                if v != usuario and w >= umbral:
                    similitud[v] += 1
                    if ("U", v) not in visitado:
                        visitado.add(("U", v))
                        cola.append(("U", v, 2))

        else:                                # nivel 2: vecino -> películas nuevas (nivel 3)
            if vecinos is None:
                # Por ser FIFO, cuando sale el primer nodo del nivel 2 ya se procesó todo el nivel 1
                # y las similitudes están completas: nos quedamos con los max_vecinos más parecidos.
                # Empate: gana el id menor, para que el resultado sea siempre el mismo.
                n = len(similitud) if max_vecinos is None else max_vecinos
                vecinos = dict(heapq.nlargest(n, similitud.items(), key=lambda p: (p[1], -p[0])))
            if x not in vecinos:
                continue
            for m, w in grafo.adj_usuario[x].items():
                stats["aristas"] += 1
                if w >= umbral and m not in vistas:   # el nivel 3 no se encola: ahí termina el recorrido
                    puntaje[m] += vecinos[x] * w

    stats["candidatos"] = len(puntaje)
    return dict(puntaje), (vecinos or {}), stats

In [ ]:
# Celda 6.2: selección voraz Top-K con min-heap
def top_k(puntajes, k=K):
    heap = []                                 # min-heap: heap[0] es el peor de los K mejores
    for pelicula, p in puntajes.items():
        item = (p, -pelicula)                 # empate de puntaje: gana el id de película menor
        if len(heap) < k:
            heapq.heappush(heap, item)        # O(log K)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)     # saca el peor y mete el nuevo, O(log K)
    return [(-neg_id, p) for p, neg_id in sorted(heap, reverse=True)]


def recomendar(usuario, k=K, max_vecinos=MAX_VECINOS, grafo=None):
    grafo = grafo or G
    puntajes, vecinos, stats = bfs_candidatos(grafo, usuario, max_vecinos=max_vecinos)
    return top_k(puntajes, k), vecinos, stats


def tabla_recomendaciones(recs):
    return pd.DataFrame([{"Puesto": i, "Película": titulo.get(m, m), "Géneros": ", ".join(generos_de.get(m, [])),
                          "Puntaje": round(p, 1)} for i, (m, p) in enumerate(recs, start=1)]).set_index("Puesto")

In [ ]:
# Celda 6.3: ejemplo con el usuario de demostración
t0 = time.perf_counter()
recs, vecinos_demo, stats_demo = recomendar(USUARIO_DEMO)
t_demo = (time.perf_counter() - t0) * 1000

le_gustaron = sorted(((w, m) for m, w in G.adj_usuario[USUARIO_DEMO].items() if w >= GUSTO), reverse=True)
print(f"Al usuario {USUARIO_DEMO} le gustaron {len(le_gustaron)} películas, por ejemplo:")
for w, m in le_gustaron[:5]:
    print(f"   {w}★  {titulo[m]}")
print(f"\nBFS: {stats_demo['nodos']:,} nodos visitados, {stats_demo['aristas']:,} aristas revisadas, "
      f"{len(vecinos_demo)} vecinos, {stats_demo['candidatos']:,} películas candidatas")
print(f"Tiempo total (BFS + voraz): {t_demo:.1f} ms\n")
tabla_recomendaciones(recs)

In [ ]:
# Celda 6.4: Figura 4, el recorrido BFS del usuario de demostración (versión reducida para que se lea)
N1, N2, N3 = 5, 5, 5
top_vec = sorted(vecinos_demo.items(), key=lambda p: (-p[1], p[0]))[:N2]
recs_fig = [m for m, _ in recs[:N3]]
# nivel 1: las películas que le gustaron a u y que comparten más vecinos del dibujo
comunes = Counter(m for v, _ in top_vec for m, w in G.adj_usuario[v].items()
                  if w >= GUSTO and G.adj_usuario[USUARIO_DEMO].get(m, 0) >= GUSTO)
nivel1 = [m for m, _ in comunes.most_common(N1)]

T = nx.Graph()
T.add_node(f"U{USUARIO_DEMO}", nivel=0)
T.add_nodes_from((f"M{m}" for m in nivel1), nivel=1)
T.add_nodes_from((f"U{v}" for v, _ in top_vec), nivel=2)
T.add_nodes_from((f"R{m}" for m in recs_fig), nivel=3)   # "R": película recomendada
for m in nivel1:
    T.add_edge(f"U{USUARIO_DEMO}", f"M{m}")
    for v, _ in top_vec:
        if G.adj_usuario[v].get(m, 0) >= GUSTO:
            T.add_edge(f"M{m}", f"U{v}")
for v, _ in top_vec:
    for m in recs_fig:
        if G.adj_usuario[v].get(m, 0) >= GUSTO:
            T.add_edge(f"U{v}", f"R{m}")

pos = nx.multipartite_layout(T, subset_key="nivel")
color = {0: "#c0392b", 1: "#e08a2c", 2: "#2f6fb0", 3: "#3a9e5c"}
fig, ax = plt.subplots(figsize=(14, 7), dpi=150)
nx.draw_networkx_edges(T, pos, edge_color="#9a9a9a", alpha=0.6, ax=ax)
nx.draw_networkx_nodes(T, pos, node_color=[color[d["nivel"]] for _, d in T.nodes(data=True)], node_size=700, ax=ax)
etiquetas = {n: (n if n[0] == "U" else titulo[int(n[1:])][:22]) for n in T.nodes}
for n, (x, y) in pos.items():
    ax.text(x, y - 0.06, etiquetas[n], ha="center", va="top", fontsize=7)
for nivel, nombre in enumerate(["Nivel 0\nusuario", "Nivel 1\npelículas que le gustaron",
                                "Nivel 2\nvecinos más parecidos", "Nivel 3\nrecomendadas"]):
    xs = [pos[n][0] for n, d in T.nodes(data=True) if d["nivel"] == nivel]
    if xs:
        ax.text(xs[0], 1.02, nombre, ha="center", fontsize=9, weight="bold", transform=ax.get_xaxis_transform())
ax.set_title(f"Figura 4. Recorrido BFS de 3 niveles desde el usuario {USUARIO_DEMO} (vista reducida)", fontsize=11, pad=50)
ax.axis("off")
fig.savefig(f"{SALIDA}/figura4_recorrido_bfs.png", bbox_inches="tight")
plt.show()

## Parte 7. Fuerza bruta (línea base para validar)

Compara al usuario con **todos** los demás usuarios y puntúa **todas** las películas no vistas, luego ordena la lista completa.
Costo O(|U|·|M|) más O(|M| log |M|) del ordenamiento. Es lenta, pero sirve como referencia: si el BFS no descarta vecinos (`max_vecinos=None`), los dos métodos deben dar **exactamente el mismo ranking**.

In [ ]:
# Celda 7.1: recomendación por fuerza bruta
def recomendar_fuerza_bruta(grafo, usuario, k=K, umbral=GUSTO):
    vistas = grafo.adj_usuario.get(usuario, {})
    gustos_u = {m for m, w in vistas.items() if w >= umbral}
    puntaje = {}
    for v, calificadas in grafo.adj_usuario.items():          # TODOS los usuarios
        if v == usuario:
            continue
        sim = sum(1 for m in gustos_u if calificadas.get(m, 0) >= umbral)
        if sim == 0:
            continue
        for m in grafo.adj_pelicula:                           # TODAS las películas
            w = calificadas.get(m, 0)
            if w >= umbral and m not in vistas:
                puntaje[m] = puntaje.get(m, 0.0) + sim * w
    ranking = sorted(puntaje.items(), key=lambda p: (-p[1], p[0]))   # ordena todo: O(C log C)
    return ranking[:k]


# Comprobación: BFS sin límite de vecinos + voraz == fuerza bruta (en 3 usuarios, uno por integrante)
usuarios_fb = [elegir(a, b, 1, semilla=SEMILLA + 1)[0] for a, b in RANGOS.values()]
filas = []
for u in usuarios_fb:
    t0 = time.perf_counter(); fb = recomendar_fuerza_bruta(G, u); t_fb = time.perf_counter() - t0
    t0 = time.perf_counter(); bv, _, _ = recomendar(u, max_vecinos=None); t_bv = time.perf_counter() - t0
    t0 = time.perf_counter(); b50, _, _ = recomendar(u); t_b50 = time.perf_counter() - t0
    assert fb == bv, f"El ranking no coincide para el usuario {u}"
    filas.append({"Usuario": u, "Fuerza bruta (s)": round(t_fb, 3), "BFS sin límite + voraz (s)": round(t_bv, 3),
                  f"BFS N={MAX_VECINOS} + voraz (s)": round(t_b50, 3), "Mismo Top-K que fuerza bruta": fb == bv,
                  f"Coincidencias Top-{K} con N={MAX_VECINOS}": len({m for m, _ in fb} & {m for m, _ in b50})})
tabla_fb = pd.DataFrame(filas).set_index("Usuario")
tabla_fb.to_csv(f"{SALIDA}/fuerza_bruta_vs_bfs.csv")
print("OK: BFS + voraz reproduce exactamente el ranking de la fuerza bruta.")
tabla_fb

## Parte 8. Comunidades de gustos: UFDS + Kruskal

**UFDS (Union-Find).** Cada usuario empieza en su propio conjunto. `union(u, v)` junta dos conjuntos y `find(u)` devuelve el representante. Con **compresión de caminos** y **unión por rango**, cada operación cuesta O(α(n)), prácticamente constante.

1. **Componentes conexas.** Uniendo los extremos de todas las aristas usuario–usuario se obtienen los grupos conectados para cada umbral S.
2. **Kruskal con tope de tamaño.** Las componentes conexas suelen dar un grupo gigante (los usuarios muy activos se conectan con casi todos), así que no sirven como comunidades. Por eso se aplica Kruskal: se ordenan las aristas usuario–usuario por **similitud de Jaccard** (películas que les gustaron a ambos / películas que le gustaron a alguno de los dos) de mayor a menor y se recorren una por una. UFDS decide en O(α(n)) si los dos usuarios ya están en el mismo grupo (la arista formaría un ciclo y se descarta); si no, se unen, **salvo que el grupo resultante pase de `TAMANO_MAX_COMUNIDAD` usuarios**. Así primero se juntan los usuarios más parecidos y ningún grupo absorbe a todos. Ordenar cuesta O(|E| log |E|) y el recorrido O(|E| · α(n)).

In [ ]:
# Celda 8.1: UFDS con compresión de caminos y unión por rango
class UFDS:
    def __init__(self, elementos):
        self.padre = {x: x for x in elementos}
        self.rango = {x: 0 for x in elementos}
        self.tamano = {x: 1 for x in elementos}   # usuarios en el grupo de cada raíz
        self.componentes = len(self.padre)

    def find(self, x):
        raiz = x
        while self.padre[raiz] != raiz:          # 1) subir hasta la raíz
            raiz = self.padre[raiz]
        while self.padre[x] != raiz:             # 2) compresión: todos apuntan directo a la raíz
            self.padre[x], x = raiz, self.padre[x]
        return raiz

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                         # ya estaban en el mismo grupo
        if self.rango[ra] < self.rango[rb]:      # unión por rango: el árbol bajo cuelga del alto
            ra, rb = rb, ra
        self.padre[rb] = ra
        self.tamano[ra] += self.tamano[rb]
        if self.rango[ra] == self.rango[rb]:
            self.rango[ra] += 1
        self.componentes -= 1
        return True

    def grupos(self):
        g = defaultdict(list)
        for x in self.padre:
            g[self.find(x)].append(x)
        return sorted(g.values(), key=len, reverse=True)


# Prueba pequeña a mano: {1,2,3} y {4,5} y {6}
uf = UFDS(range(1, 7)); uf.union(1, 2); uf.union(2, 3); uf.union(4, 5)
assert uf.componentes == 3 and uf.find(3) == uf.find(1) and uf.find(4) != uf.find(1)
assert uf.tamano[uf.find(1)] == 3 and not uf.union(1, 3)
print("UFDS OK:", uf.grupos())

In [ ]:
# Celda 8.2: componentes conexas de la red usuario–usuario según el umbral S
filas = []
for s in (10, 20, 30, 50, 75, 100):
    a, b, w = aristas_usuario_usuario(s)
    uf = UFDS(ids_u.tolist())
    for x, y in zip(a.tolist(), b.tolist()):
        uf.union(x, y)
    tam = [len(g) for g in uf.grupos()]
    filas.append({"S": s, "Aristas usuario–usuario": len(w), "Componentes": uf.componentes,
                  "Componente mayor": tam[0], "Usuarios aislados": sum(t == 1 for t in tam),
                  "Grupos de 2 o más": sum(t >= 2 for t in tam)})
tabla_s = pd.DataFrame(filas).set_index("S")
tabla_s.to_csv(f"{SALIDA}/componentes_por_umbral.csv")
tabla_s

In [ ]:
# Celda 8.3: Kruskal con tope de tamaño + UFDS (similitud de Jaccard sobre las aristas con peso >= S)
jacc = uu_w / (me_gusta[np.searchsorted(ids_u, uu_a)] + me_gusta[np.searchsorted(ids_u, uu_b)] - uu_w)
orden = np.argsort(-jacc, kind="stable")             # aristas de mayor a menor similitud: O(E log E)

conectados = sorted(set(uu_a.tolist()) | set(uu_b.tolist()))
uf = UFDS(conectados)
aceptadas = 0                                        # aristas que acepta Kruskal (forman un bosque)
t0 = time.perf_counter()
for a, b in zip(uu_a[orden].tolist(), uu_b[orden].tolist()):
    ra, rb = uf.find(a), uf.find(b)
    if ra == rb:
        continue                                     # ya están en el mismo grupo: la arista haría un ciclo
    if uf.tamano[ra] + uf.tamano[rb] > TAMANO_MAX_COMUNIDAD:
        continue                                     # el grupo quedaría demasiado grande
    uf.union(a, b)
    aceptadas += 1
t_kruskal = time.perf_counter() - t0

comunidades = [g for g in uf.grupos() if len(g) >= MIN_TAMANO_COMUNIDAD]
en_comunidad = sum(len(g) for g in comunidades)
print(f"Kruskal revisó {len(orden):,} aristas y aceptó {aceptadas:,} en {t_kruskal:.1f} s")
print(f"Comunidades de {MIN_TAMANO_COMUNIDAD} a {TAMANO_MAX_COMUNIDAD} usuarios: {len(comunidades)}, "
      f"con {en_comunidad:,} usuarios ({100 * en_comunidad / len(ids_u):.1f} % del total)")
print("Tamaños:", [len(g) for g in comunidades][:30], "..." if len(comunidades) > 30 else "")

In [ ]:
# Celda 8.4: perfil de cada comunidad (géneros que más le gustan comparados con el promedio)
def perfil_generos(usuarios):
    c = Counter()
    for u in usuarios:
        for m, w in G.adj_usuario[u].items():
            if w >= GUSTO:
                for g in generos_de.get(m, []):
                    c[g] += 1
    total = sum(c.values()) or 1
    return {g: n / total for g, n in c.items()}


perfil_global = perfil_generos(G.adj_usuario.keys())
filas = []
for i, grupo in enumerate(comunidades[:15], start=1):          # las 15 más grandes
    p = perfil_generos(grupo)
    lift = sorted(((p[g] / perfil_global[g], g) for g in p if perfil_global.get(g)), reverse=True)[:3]
    edades = users.set_index("UserID").loc[grupo, "Age"]
    filas.append({"Comunidad": i, "Usuarios": len(grupo),
                  "Géneros más preferidos (veces sobre el promedio)": ", ".join(f"{g} ({l:.2f}×)" for l, g in lift),
                  "Edad más común": edades.mode().iat[0]})
tabla_com = pd.DataFrame(filas).set_index("Comunidad")
tabla_com.to_csv(f"{SALIDA}/comunidades.csv")

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(range(1, len(comunidades) + 1), [len(g) for g in comunidades], color="#8e5bb5")
ax.set(title=f"Figura 5. Tamaño de las {len(comunidades)} comunidades encontradas (Kruskal + UFDS, S = {S})",
       xlabel="Comunidad (de mayor a menor)", ylabel="Usuarios")
fig.tight_layout(); fig.savefig(f"{SALIDA}/figura5_comunidades.png", dpi=200); plt.show()
tabla_com

## Parte 9. Validación de resultados y pruebas

1. **Pruebas unitarias** sobre un grafo pequeño hecho a mano, con la respuesta calculada en papel.
2. **Precisión fuera de muestra (hold-out).** A cada usuario de una muestra se le **ocultan** el 20 % de las películas que le gustaron; se recomienda con el resto y se mide cuántas ocultas aparecen en el Top-K:
   - Precision@K = aciertos / K
   - Recall@K = aciertos / películas ocultas
   - Tasa de acierto = % de usuarios con al menos un acierto

   Se compara contra una línea base sencilla: recomendar las películas **más populares** que el usuario no vio.
3. **Tiempos y complejidad empírica:** tiempo del BFS frente al número de aristas que revisa, y heap frente a ordenar todo.

In [ ]:
# Celda 9.1: pruebas unitarias con un grafo pequeño (resultado calculado a mano)
g = GrafoBipartito()
for u, m, r in [(1, 10, 5), (1, 11, 4), (1, 12, 2),          # al usuario 1 le gustan 10 y 11
                (2, 10, 5), (2, 11, 5), (2, 13, 5),          # 2 comparte 10 y 11 -> similitud 2
                (3, 10, 4), (3, 14, 4), (3, 13, 1),          # 3 comparte 10      -> similitud 1
                (4, 15, 5)]:                                  # 4 no comparte nada -> no es vecino
    g.agregar_arista(u, m, r)

p, vec, _ = bfs_candidatos(g, 1)
assert vec == {2: 2, 3: 1}                     # vecinos y su similitud
assert p == {13: 2 * 5, 14: 1 * 4}             # 13: sim(2)·5 = 10 ; 14: sim(3)·4 = 4 (el 1★ de 3 no cuenta)
assert top_k(p, 1) == [(13, 10.0)]             # voraz: el de mayor puntaje
assert top_k({7: 3.0, 5: 3.0, 9: 1.0}, 2) == [(5, 3.0), (7, 3.0)]   # empate -> id menor primero
assert recomendar_fuerza_bruta(g, 1, k=5) == top_k(p, 5)
assert 12 not in p and 10 not in p             # nunca recomienda algo que ya vio
p1, _, _ = bfs_candidatos(g, 1, max_vecinos=1)
assert p1 == {13: 10}                          # con N = 1 solo se usa el vecino más parecido
assert bfs_candidatos(g, 999)[0] == {}         # usuario sin calificaciones: no hay candidatos
print("Todas las pruebas unitarias pasaron.")

In [ ]:
# Celda 9.2: precisión fuera de muestra (hold-out del 20 %) contra la línea base de popularidad
N_EVAL = 300                 # usuarios evaluados (súbelo para más precisión; tarda más)
OCULTAR = 0.2
rng = random.Random(SEMILLA)
populares = [m for m, _ in sorted(G.adj_pelicula.items(), key=lambda p: (-sum(w >= GUSTO for w in p[1].values()), p[0]))]

muestra = rng.sample(sorted(G.adj_usuario), N_EVAL)
res = defaultdict(list)
for u in muestra:
    gustos_u = sorted(m for m, w in G.adj_usuario[u].items() if w >= GUSTO)
    if len(gustos_u) < 10:
        continue
    ocultas = set(rng.sample(gustos_u, max(1, int(OCULTAR * len(gustos_u)))))
    guardadas = {m: G.adj_usuario[u][m] for m in ocultas}
    for m in ocultas:
        G.quitar_arista(u, m)                     # el sistema ya no "sabe" que las vio
    try:
        rec_bfs = {m for m, _ in recomendar(u)[0]}
        rec_pop = [m for m in populares if m not in G.adj_usuario[u]][:K]
        for nombre, rec in (("BFS + voraz", rec_bfs), ("Más populares", set(rec_pop))):
            aciertos = len(rec & ocultas)
            res[(nombre, "Precision@K")].append(aciertos / K)
            res[(nombre, "Recall@K")].append(aciertos / len(ocultas))
            res[(nombre, "Tasa de acierto")].append(aciertos > 0)
    finally:
        for m, w in guardadas.items():            # se devuelve el grafo a su estado original
            G.agregar_arista(u, m, w)

tabla_eval = pd.DataFrame({n: {met: round(100 * np.mean(v), 1) for (n2, met), v in res.items() if n2 == n}
                           for n in ("BFS + voraz", "Más populares")}).T
tabla_eval.columns = [f"{c} (%)" for c in tabla_eval.columns]
tabla_eval.to_csv(f"{SALIDA}/validacion_precision.csv")
print(f"{len(res[('BFS + voraz', 'Recall@K')])} usuarios evaluados, K = {K}, se ocultó el {int(OCULTAR*100)} % de sus 'me gusta'")
tabla_eval

In [ ]:
# Celda 9.3: Figura 6, complejidad empírica
# (a) tiempo del BFS frente a las aristas que revisa: si es O(|V'| + |E'|), los puntos forman una recta
medidas = []
for u in rng.sample(sorted(G.adj_usuario), 200):
    t0 = time.perf_counter()
    p, _, st = bfs_candidatos(G, u)
    medidas.append((st["aristas"], (time.perf_counter() - t0) * 1000, len(p)))
med = pd.DataFrame(medidas, columns=["aristas", "ms", "candidatos"])

# (b) voraz con heap O(C log K) frente a ordenar todos los candidatos O(C log C)
p_grande = max((bfs_candidatos(G, u)[0] for u in usuarios_fb), key=len)
ks = [1, 5, 10, 50, 100, 500]
t_heap, t_sort = [], []
for k in ks:
    t0 = time.perf_counter(); [top_k(p_grande, k) for _ in range(20)]; t_heap.append((time.perf_counter() - t0) / 20 * 1000)
    t0 = time.perf_counter(); [sorted(p_grande.items(), key=lambda x: -x[1])[:k] for _ in range(20)]; t_sort.append((time.perf_counter() - t0) / 20 * 1000)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].scatter(med.aristas, med.ms, s=10, color="#2f6fb0")
coef = np.polyfit(med.aristas, med.ms, 1)
xs = np.linspace(0, med.aristas.max(), 50)
ax[0].plot(xs, np.polyval(coef, xs), color="#c0392b", label="ajuste lineal")
ax[0].set(title="BFS: tiempo vs. aristas revisadas (200 usuarios)", xlabel="Aristas revisadas", ylabel="ms")
ax[0].legend()
ax[1].plot(ks, t_heap, "o-", label="Voraz con heap, O(C log K)")
ax[1].plot(ks, t_sort, "s-", label="Ordenar todo, O(C log C)")
ax[1].set(title=f"Top-K sobre {len(p_grande):,} candidatos", xlabel="K", ylabel="ms", xscale="log")
ax[1].legend()
fig.suptitle("Figura 6. Complejidad empírica de la propuesta", fontsize=12)
fig.tight_layout(); fig.savefig(f"{SALIDA}/figura6_tiempos.png", dpi=200); plt.show()

r = np.corrcoef(med.aristas, med.ms)[0, 1]
print(f"BFS + voraz: promedio {med.ms.mean():.1f} ms por usuario (máximo {med.ms.max():.1f} ms); "
      f"correlación tiempo–aristas r = {r:.3f}")
print("Nota: sorted() de Python está programado en C y el heap de top_k en Python, por eso para K grande "
      "ordenar todo puede ganar en tiempo aunque su complejidad sea mayor.")
i10 = ks.index(10)
print(f"Con K = 10: heap {t_heap[i10]:.3f} ms vs. ordenar todo {t_sort[i10]:.3f} ms")

## Parte 10. Interfaz gráfica (GUI)

El enunciado pide demostrar la aplicación con una interfaz gráfica. Esta celda crea una GUI con **ipywidgets** que funciona dentro de Colab: se elige el usuario, K y el número de vecinos, y al presionar **Recomendar** muestra lo que le gustó al usuario, las K películas recomendadas, su comunidad y el tiempo de respuesta.

(El repositorio también tiene una versión de escritorio en Tkinter: `python main.py --gui`.)

In [ ]:
# Celda 10.1: GUI en Colab con ipywidgets
comunidad_de = {u: i for i, grupo in enumerate(comunidades, start=1) for u in grupo}

try:
    import ipywidgets as w
    from IPython.display import display, clear_output

    usuario_w = w.BoundedIntText(value=USUARIO_DEMO, min=int(ids_u.min()), max=int(ids_u.max()), description="Usuario:")
    k_w = w.IntSlider(value=K, min=1, max=30, description="K:")
    vec_w = w.IntSlider(value=MAX_VECINOS, min=5, max=300, step=5, description="Vecinos N:")
    boton = w.Button(description="Recomendar", button_style="danger", icon="film")
    salida = w.Output()

    def al_presionar(_):
        with salida:
            clear_output()
            u = usuario_w.value
            if u not in G.adj_usuario:
                print("Ese usuario no existe en el dataset."); return
            t0 = time.perf_counter()
            recs_u, vec_u, st = recomendar(u, k=k_w.value, max_vecinos=vec_w.value)
            ms = (time.perf_counter() - t0) * 1000
            info = users.set_index("UserID").loc[u]
            print(f"Usuario {u} | sexo {info.Gender} | edad {info.Age} | {len(G.adj_usuario[u])} calificaciones "
                  f"| comunidad {comunidad_de.get(u, 'sin comunidad (aislado)')}")
            print("Le gustaron, por ejemplo:", "; ".join(titulo[m] for m, r in
                  sorted(G.adj_usuario[u].items(), key=lambda p: -p[1])[:5]))
            print(f"BFS revisó {st['aristas']:,} aristas, usó {len(vec_u)} vecinos y evaluó {st['candidatos']:,} "
                  f"candidatas en {ms:.1f} ms\n")
            display(tabla_recomendaciones(recs_u))

    boton.on_click(al_presionar)
    display(w.HTML("<h3>🎬 Recomendador Grupo 4 · MovieLens 1M</h3>"),
            w.HBox([usuario_w, k_w]), w.HBox([vec_w, boton]), salida)
    al_presionar(None)
except ImportError:
    print("ipywidgets no está instalado: en Colab ya viene incluido.")

## Parte 11. Cifras para el informe y descarga de resultados

In [ ]:
# Celda 11.1: resumen de cifras (cópialas al informe)
cifras = {
    "Calificaciones (aristas usuario–película)": f"{G.num_aristas():,}",
    "Nodos activos": f"{G.num_nodos():,} ({G.num_usuarios():,} usuarios + {G.num_peliculas():,} películas)",
    "Nodos contando películas sin calificación": f"{len(users) + len(movies):,}",
    "Densidad bipartita": f"{G.densidad():.4f}",
    f"Aristas usuario–usuario (S = {S})": f"{len(uu_w):,}",
    "Comunidades (Kruskal + UFDS)": f"{len(comunidades)}, con {en_comunidad:,} usuarios en total",
    "Tiempo promedio de una recomendación": f"{med.ms.mean():.1f} ms",
    f"Precision@{K} BFS + voraz / populares": f"{tabla_eval.iloc[0, 0]} % / {tabla_eval.iloc[1, 0]} %",
    f"Recall@{K} BFS + voraz / populares": f"{tabla_eval.iloc[0, 1]} % / {tabla_eval.iloc[1, 1]} %",
    "Datos": "SINTÉTICOS (no usar en el informe)" if USAR_DATOS_SINTETICOS else "MovieLens 1M reales",
}
pd.Series(cifras).to_frame("Valor").to_csv(f"{SALIDA}/cifras_informe.csv")
pd.Series(cifras).to_frame("Valor")

In [ ]:
# Celda 11.2: comprimir figuras y tablas en resultados_grupo4.zip y descargarlo
with zipfile.ZipFile("resultados_grupo4.zip", "w") as z:
    for f in sorted(os.listdir(SALIDA)):
        z.write(f"{SALIDA}/{f}", f)
print("Contenido:", sorted(os.listdir(SALIDA)))
try:
    from google.colab import files
    files.download("resultados_grupo4.zip")
except ImportError:
    print("Fuera de Colab: el zip quedó en la carpeta actual.")

## Referencias

- Harper, F. M., & Konstan, J. A. (2015). The MovieLens datasets: History and context. *ACM Transactions on Interactive Intelligent Systems, 5*(4), Article 19. https://doi.org/10.1145/2827872
- Demović, L., et al. (2013). Movie recommendation based on graph traversal algorithms. *24th International Workshop on Database and Expert Systems Applications*, 152–156. https://doi.org/10.1109/DEXA.2013.24
- Zhou, T., Ren, J., Medo, M., & Zhang, Y.-C. (2007). Bipartite network projection and personal recommendation. *Physical Review E, 76*(4), 046115. https://doi.org/10.1103/PhysRevE.76.046115
- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). *Introduction to Algorithms* (4.ª ed.). MIT Press. (BFS, heaps, Union-Find y Kruskal)